In [ ]:
!pip install -q py-feat
!apt-get -qq install -y ffmpeg

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ThesisExperiment_cleann'
os.chdir(BASE_DIR)

missing = []
for pid in range(1, 7):
    for f in [f"participant_{pid}_continuous_ppi.csv",
              f"participant_{pid}_godspeed_scores.csv"]:
        if not os.path.exists(f):
            missing.append(f)
    for robot in ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]:
        vid = f"participant_{pid}_{robot}_cam.avi"
        if not os.path.exists(vid):
            missing.append(vid)

if missing:
    print(f"MISSING {len(missing)} file(s):")
    for m in missing:
        print(" -", m)
else:
    print("All expected files found. Ready to run.")

In [ ]:
"""
IndividualAnalysis.py -- adapted for Colab + Py-Feat Detectorv2, all 6 participants.

Changes from the original (Detectorv1 / local Mac, single-participant) version:
  - Detectorv2 instead of Detectorv1: one hardcoded RetinaFace + multitask
    network instead of separately swappable face/landmark/AU/emotion models.
    No face_model / landmark_model / au_model / emotion_model / n_jobs kwargs --
    v2 doesn't take them.
  - .detect(path, data_type="video", skip_frames=...) instead of
    .detect_video(path, batch_size=..., every_n_frame=...) -- v1's method
    name and kwargs don't exist on v2.
  - device="cuda" (Colab GPU) instead of "cpu". Requires
    torch.backends.cudnn.enabled = False (see setup below) to avoid the
    "GET was unable to find an engine to execute this computation" error
    hit on Colab's T4 -- a known PyTorch/cuDNN kernel-selection issue, not
    a Py-Feat bug.
  - identity_model=None: v2 computes a 512-d ArcFace embedding per frame by
    default. You already track identity via PARTICIPANT_ID/filename, so
    this is switched off to keep output size sane across 30 videos.
  - TARGET_AUS fixed to Py-Feat's real zero-padded column names (AU04, not
    AU4). The original script's AU4/AU5/AU7/AU9 never matched any column in
    either v1 or v2's output -- this was a silent bug independent of which
    detector you use, and would have produced empty AU columns either way.
  - TARGET_AUS now covers all 20 AUs Py-Feat produces (not just 4/5/7/9),
    plus arousal, valence, and the 7-class emotion probabilities, per your
    request to keep the full AU set + affect columns rather than just your
    four original AUs of interest. AU04/05/07/09 remain in this set --
    nothing is lost, it's a superset.
  - EXTRA_AFFECT_COLS (arousal, valence, emotions) added as a second column
    group alongside the AUs, carried through the same merge/rename logic.
  - Google Drive mount + BASE_DIR: reads/writes files from your Drive
    instead of a local folder, since Colab can't see your Mac's filesystem.
  - Wrapped in `for PARTICIPANT_ID in PARTICIPANT_IDS:` so all 6
    participants (30 videos total) run in one execution instead of one
    script run per participant. The detector is created once, outside the
    loop, and reused across all participants/trials -- re-creating it per
    participant would reload the model weights 6x for no reason.

Everything else -- PPI/RMSSD handling, Godspeed processing, camera-offset
timestamp correction, cross-participant exclusion flagging, per-participant
output file structure -- is UNCHANGED from the original script's logic, just
indented one level to live inside the participant loop.
"""

import pandas as pd
import os
import subprocess
import torch
from feat import Detectorv2
from google.colab import drive

# ==========================================
# GOOGLE DRIVE SETUP
# ==========================================
# Mounts your Drive at /content/drive. First run will prompt a browser
# sign-in/permission popup -- approve it. BASE_DIR should point at the
# folder inside your Drive containing all the participant videos and CSVs
# (the same files that lived in ThesisExperiment_cleann on your Mac).
# Adjust the path below to match wherever you actually upload that folder.
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ThesisExperiment_cleann'
os.chdir(BASE_DIR)
print(f"Working directory set to: {os.getcwd()}")

# ==========================================
# COLAB / GPU SETUP
# ==========================================
# Fixes "GET was unable to find an engine to execute this computation" on
# Colab's T4 GPU -- a cuDNN kernel-selection issue, confirmed reproducible
# and confirmed fixed by this flag in testing. Must be set before the
# detector is constructed.
torch.backends.cudnn.enabled = False

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# CONFIGURATION
# ==========================================
ROBOT_TRIALS = ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]
PARTICIPANT_IDS = [1, 2, 3, 4, 5, 6]

# Trials that should NOT be pooled with the rest of the sample in any
# cross-participant comparison, even though the participant's own data is
# otherwise valid and should be kept. Documented in Limitations.docx
# ("Stimulus Design"). Keyed by (participant_id, robot_name); the row for
# that trial is still produced below (not deleted), just flagged, so a
# future cross-participant aggregation script can filter it out with one
# line rather than relying on someone remembering this paragraph.
CROSS_PARTICIPANT_EXCLUSIONS = {
    (1, "Robot4"): "Used a different, uncut stimulus video showing three "
                   "distinct robots in sequence, vs. the edited two-robot "
                   "version (one robot removed from the middle) used for all "
                   "other participants -- both the video content and the "
                   "resulting Godspeed ratings differ, not just the duration, "
                   "so this trial is not comparable to any other participant's "
                   "Robot4 trial.",
}

# All 20 AUs Py-Feat's Detectorv2 produces, using its real zero-padded column
# names. AU04/05/07/09 (the thesis's original four AUs of interest) are
# included here -- this is a superset, not a replacement.
TARGET_AUS = [
    "AU01", "AU02", "AU04", "AU05", "AU06", "AU07", "AU09", "AU10", "AU11",
    "AU12", "AU14", "AU15", "AU17", "AU20", "AU23", "AU24", "AU25", "AU26",
    "AU28", "AU43",
]

# Affect columns unique to Detectorv2 -- not part of Detectorv1's output.
# arousal/valence: continuous, from v2's dedicated V/A head (not derived
# from the AUs). Emotion columns: v2's 7-class softmax probabilities.
EXTRA_AFFECT_COLS = [
    "arousal", "valence",
    "Neutral", "Happy", "Sad", "Surprise", "Fear", "Disgust", "Anger",
]

# Py-Feat's confidence column is typically "FaceScore" (0-1), not OpenFace's binary
# "success". 0.75 is the cutoff used in a published EMG-validation study comparing
# OpenFace/Py-Feat/FaceReader AU accuracy; adjust if you have a reason to.
FACE_CONFIDENCE_THRESHOLD = 0.75

# How many frames Py-Feat actually processes per second of video (source is ~30fps).
#   1  -> every frame  (~30Hz, finest detail, slowest)
#   5  -> ~6Hz         (reasonable middle ground -- current default)
#   30 -> ~1Hz         (fastest, coarsest -- what this script used before)
FRAME_SAMPLE_INTERVAL = 5

# Nominal capture fps set in MasterScriptTest.py's cv2.VideoWriter. Cross-check this
# against the "X frames written" console line MasterScriptTest.py prints for each
# trial vs. that trial's actual video_playback duration (from the PPI CSV) -- if the
# real fps drifted from 30 (common with OpenCV on macOS under load), update this.
ACTUAL_FPS = 30.0

# Max allowed gap (seconds) between a PPI sample and its nearest facial-AU sample in
# the merge below. Without this, merge_asof(direction="nearest") will happily pair
# samples that are seconds apart. Widen if too many rows end up with NaN AUs; narrow
# if matches look too loose.
MERGE_TOLERANCE_SEC = 1.0


def compute_rmssd(ppi_values_ms):
    """RMSSD = sqrt(mean of squared successive differences between consecutive
    PPI values, in ms). Needs at least 2 values (1 successive difference) to
    compute anything; returns None if there isn't enough data."""
    values = pd.Series(ppi_values_ms).dropna().values
    if len(values) < 2:
        return None
    successive_diffs = values[1:] - values[:-1]
    return (successive_diffs ** 2).mean() ** 0.5


def process_godspeed_scores(scores_df, target_participant_id, godspeed_file):
    subscales = {
        "anthropomorphism": ["anthropomorphism_1", "anthropomorphism_2", "anthropomorphism_3", "anthropomorphism_4", "anthropomorphism_5"],
        "animacy": ["animacy_1", "animacy_2", "animacy_3", "animacy_4", "animacy_5", "animacy_6"],
        "likeability": ["likeability_1", "likeability_2", "likeability_3", "likeability_4", "likeability_5"],
        "perceived_intelligence": ["intelligence_1", "intelligence_2", "intelligence_3", "intelligence_4", "intelligence_5"],
        "perceived_safety": ["safety_1", "safety_2", "safety_3"]
    }

    # No reverse-scoring is needed here: every Godspeed item, including all
    # three Perceived Safety items, is displayed with its negative pole on the
    # left and positive pole on the right (see the GODSPEED_DIMENSIONS comment
    # in MasterScript_Final.py), so a plain average of raw 1-5 scores already
    # points the right direction (higher = more of the positive trait) for
    # every subscale.

    scores_df = scores_df.copy()

    # The filename (participant_{ID}_godspeed_scores.csv) is treated as ground truth
    # for whose data this is. If the file's own participant_id column disagrees,
    # that's almost certainly a mislabeled/misentered ID during collection -- warn
    # loudly instead of silently filtering everything out.
    if "participant_id" in scores_df.columns:
        found_ids = set(scores_df["participant_id"].unique())
        if found_ids != {target_participant_id}:
            print(f"  [!] WARNING: {godspeed_file} is internally labeled with participant_id "
                  f"{found_ids}, not {target_participant_id} (from the filename). Treating all "
                  f"rows as participant {target_participant_id}'s data -- double-check this is "
                  f"actually the right participant's file.")
    scores_df["participant_id"] = target_participant_id

    processed_dfs = []
    for robot_name, group in scores_df.groupby("robot_name"):
        if len(group) > 1:
            print(f"  [!] WARNING: {len(group)} Godspeed rows found for participant "
                  f"{target_participant_id}, robot {robot_name} -- using the first one.")
        row_data = {"participant_id": target_participant_id, "robot_name": robot_name}
        for subscale_name, cols in subscales.items():
            valid_cols = [c for c in cols if c in group.columns]
            if valid_cols:
                row_data[subscale_name] = group[valid_cols].mean(axis=1).values[0]
            else:
                row_data[subscale_name] = None
        processed_dfs.append(pd.DataFrame([row_data]))

    if not processed_dfs:
        return pd.DataFrame()

    return pd.concat(processed_dfs, ignore_index=True)


# Initialize Py-Feat Detectorv2 once, globally, on GPU -- shared across all
# 6 participants / 30 videos so the model weights are only loaded once.
# v2 hardcodes RetinaFace for face detection and runs one multitask network
# for AUs/emotion/valence-arousal/gaze/mesh/pose -- no face_model,
# landmark_model, au_model, emotion_model, or n_jobs kwargs (those are
# Detectorv1-only). identity_model=None skips the 512-d ArcFace embedding
# since it's not used downstream here.
print("Initializing Py-Feat Detectorv2 (GPU mode)...")
detector = Detectorv2(device="cuda", identity_model=None)

# ==========================================
# MAIN LOOP -- one pass per participant
# ==========================================
for PARTICIPANT_ID in PARTICIPANT_IDS:
    print(f"\n{'='*70}")
    print(f"--- Processing Individual Analysis & Py-Feat Extraction for Participant {PARTICIPANT_ID} ---")
    print(f"{'='*70}")

    godspeed_file = f"participant_{PARTICIPANT_ID}_godspeed_scores.csv"
    ppi_file = f"participant_{PARTICIPANT_ID}_continuous_ppi.csv"
    timing_log_filename = f"participant_{PARTICIPANT_ID}_camera_timing.csv"

    try:
        ppi_df = pd.read_csv(ppi_file)
        scores_df = pd.read_csv(godspeed_file)
    except FileNotFoundError as e:
        print(f"Error loading base files for participant {PARTICIPANT_ID}: {e}")
        print(f"  [!] Skipping participant {PARTICIPANT_ID} entirely.")
        continue

    # Wall-clock moment the webcam actually started recording each trial (written by the
    # updated MasterScriptTest.py). Used below to correct for the gap between when the
    # video_playback phase clock resets to 0 and when frame 0 of the webcam footage was
    # actually captured -- these are NOT the same instant (camera init takes >1.5s).
    camera_timing = {}
    if os.path.exists(timing_log_filename):
        timing_df = pd.read_csv(timing_log_filename)
        camera_timing = dict(zip(timing_df["robot_name"], timing_df["camera_start_time"]))
    else:
        print(f"  [!] WARNING: {timing_log_filename} not found -- facial timestamps will NOT be "
              f"corrected for the camera-start offset, and will assume frame 0 == video_playback "
              f"phase start.")

    # Inject participant_id and map trial_name to robot_name for merging
    ppi_df["participant_id"] = PARTICIPANT_ID
    if "trial_name" in ppi_df.columns:
        ppi_df["robot_name"] = ppi_df["trial_name"]

    # The video-playback phase clock starts the instant the phase begins in software,
    # but VLC is launched fresh each trial (cold app start + fullscreen + file open),
    # so the robot video isn't actually visible on screen for several seconds after
    # that. Since the analysis only needs per-trial averages (not fine time-locked
    # reactions to specific on-screen moments), the simplest fix is to drop this dead
    # window rather than try to detect VLC's exact playback-start moment. Set this
    # comfortably above the largest VLC launch delay you've actually observed across
    # a few pilot runs -- 4-7s was seen during testing, so 8s leaves some margin, but
    # re-check this once you have real participant data.
    STIMULUS_ONSET_TRIM_SEC = 8.0

    # Polar's own documentation for the PPI error_estimate field states values below
    # 10ms are probably very accurate, while values over 30ms may be caused by
    # movement artifact or too-loose sensor-skin contact. The device's own
    # invalid_ppi flag does NOT reliably agree with this -- real pilot data showed
    # a third of samples with error_estimate > 30ms, and 60% of those were still
    # marked invalid_ppi = False. Both filters are applied, since neither alone
    # catches everything the other does.
    PPI_ERROR_ESTIMATE_THRESHOLD_MS = 30

    # Isolate video playback phase (automatically drops camera retry flag rows)
    video_ppi = ppi_df[ppi_df["phase"] == "video_playback"].copy()
    if "invalid_ppi" in video_ppi.columns:
        video_ppi = video_ppi[video_ppi["invalid_ppi"] == False]

    if "error_estimate" in video_ppi.columns:
        before_error_filter = len(video_ppi)
        video_ppi = video_ppi[video_ppi["error_estimate"] <= PPI_ERROR_ESTIMATE_THRESHOLD_MS]
        print(f"  [i] Dropped {before_error_filter - len(video_ppi)} PPI row(s) with "
              f"error_estimate > {PPI_ERROR_ESTIMATE_THRESHOLD_MS}ms (likely movement "
              f"artifact or poor skin contact, per Polar's own PPI documentation).")

    before_trim = len(video_ppi)
    video_ppi = video_ppi[video_ppi["elapsed_trial_sec"] >= STIMULUS_ONSET_TRIM_SEC]
    print(f"  [i] Trimmed {before_trim - len(video_ppi)} PPI row(s) from the first "
          f"{STIMULUS_ONSET_TRIM_SEC}s of each trial (VLC launch dead time).")

    # Baseline-phase PPI, isolated the same way as video_ppi (same quality filters,
    # no stimulus-onset trim since there's no video/VLC delay to correct for here) --
    # this is the participant's own resting reference for each trial, used below to
    # compute a per-trial baseline RMSSD to compare against the stimulus-window RMSSD.
    baseline_ppi = ppi_df[ppi_df["phase"] == "baseline"].copy()
    if "invalid_ppi" in baseline_ppi.columns:
        baseline_ppi = baseline_ppi[baseline_ppi["invalid_ppi"] == False]
    if "error_estimate" in baseline_ppi.columns:
        baseline_ppi = baseline_ppi[baseline_ppi["error_estimate"] <= PPI_ERROR_ESTIMATE_THRESHOLD_MS]

    trial_rmssd_rows = []
    trial_integrated_dfs = []

    # ==========================================
    # STEP A: PY-FEAT EXTRACTION & TRIAL ALIGNMENT
    # ==========================================
    for robot_name in ROBOT_TRIALS:
        print(f"\nProcessing trial: {robot_name}...")

        video_filename = f"participant_{PARTICIPANT_ID}_{robot_name}_cam.avi"
        expected_openface_csv = f"participant_{PARTICIPANT_ID}_{robot_name}_openface.csv"
        temp_mp4 = f"temp_{PARTICIPANT_ID}_{robot_name}.mp4"

        # 1. AUTOMATE PY-FEAT EXTRACTION IF CSV DOES NOT EXIST YET
        if not os.path.exists(expected_openface_csv):
            if os.path.exists(video_filename):
                print(f"  [*] Converting, downscaling to 480p, and running Py-Feat on {video_filename}...")
                try:
                    subprocess.run([
                        "ffmpeg", "-y",
                        "-i", video_filename,
                        "-vf", "scale=640:-1",
                        "-c:v", "libx264",
                        "-preset", "ultrafast",
                        "-g", "30",
                        "-movflags", "+faststart",
                        temp_mp4
                    ], check=True)

                    predictions = detector.detect(
                        temp_mp4, data_type="video", skip_frames=FRAME_SAMPLE_INTERVAL,
                    )
                    predictions.to_csv(expected_openface_csv, index=False)

                    print(f"  [+] Py-Feat successfully processed {robot_name}")
                except Exception as e:
                    print(f"  [!] Warning: Py-Feat extraction failed for {robot_name}: {e}")
                finally:
                    if os.path.exists(temp_mp4):
                        os.remove(temp_mp4)
            else:
                print(f"  [!] Warning: Video file {video_filename} not found.")
        else:
            print(f"  [i] Found existing CSV for {robot_name}, skipping extraction.")

        # Isolate PPI data for this specific robot trial
        trial_ppi = video_ppi[video_ppi["robot_name"] == robot_name].copy()
        if trial_ppi.empty:
            print(f"  [!] Warning: No PPI data found for {robot_name}")
            continue

        # RMSSD for both windows -- computed here (not dependent on facial data
        # succeeding) since this is the actual physiological outcome measure the
        # baseline-vs-stimulus comparison in the Methodology chapter depends on.
        trial_baseline_ppi = baseline_ppi[baseline_ppi["robot_name"] == robot_name].sort_values("elapsed_trial_sec")
        baseline_rmssd = compute_rmssd(trial_baseline_ppi["ppi_ms"])
        stimulus_rmssd = compute_rmssd(trial_ppi.sort_values("elapsed_trial_sec")["ppi_ms"])
        if baseline_rmssd is None:
            print(f"  [!] WARNING: not enough valid baseline PPI data for {robot_name} to compute RMSSD.")
        if stimulus_rmssd is None:
            print(f"  [!] WARNING: not enough valid stimulus-window PPI data for {robot_name} to compute RMSSD.")

        exclusion_reason = CROSS_PARTICIPANT_EXCLUSIONS.get((PARTICIPANT_ID, robot_name))
        if exclusion_reason:
            print(f"  [i] NOTE: ({PARTICIPANT_ID}, {robot_name}) is flagged for cross-participant "
                  f"exclusion: {exclusion_reason}")

        trial_rmssd_rows.append({
            "participant_id": PARTICIPANT_ID,
            "robot_name": robot_name,
            "baseline_rmssd": baseline_rmssd,
            "stimulus_rmssd": stimulus_rmssd,
            "rmssd_delta": (stimulus_rmssd - baseline_rmssd) if (baseline_rmssd is not None and stimulus_rmssd is not None) else None,
            "excluded_cross_participant": exclusion_reason is not None,
            "exclusion_reason": exclusion_reason if exclusion_reason else "",
        })

        if not os.path.exists(expected_openface_csv):
            print(f"  [!] Skipping facial data merge for {robot_name} (No facial output available).")
            trial_integrated_dfs.append(trial_ppi)
            continue

        # 2. LOAD, FILTER, AND MERGE ACTION UNIT + AFFECT DATA
        openface_df = pd.read_csv(expected_openface_csv)
        openface_df.columns = openface_df.columns.str.strip()

        # Confidence filtering. Py-Feat's confidence column is "FaceScore" for
        # both Detectorv1 and Detectorv2 -- check for either name/OpenFace's
        # "success" so this doesn't silently no-op.
        conf_col = next((c for c in ["FaceScore", "face_score", "success"] if c in openface_df.columns), None)
        if conf_col:
            before = len(openface_df)
            threshold = 1 if conf_col == "success" else FACE_CONFIDENCE_THRESHOLD
            openface_df = openface_df[openface_df[conf_col] >= threshold]
            print(f"  [i] Dropped {before - len(openface_df)} low-confidence frame(s) using '{conf_col}'.")
        else:
            print(f"  [!] WARNING: no recognized confidence column in {expected_openface_csv}. "
                  f"Actual columns: {list(openface_df.columns)}")

        # Resolve the real AU + affect column names. Py-Feat's zero-padded AU
        # names (AU04, not AU4) and v2's exact-cased affect column names
        # (arousal/valence lowercase; Neutral/Happy/Sad/... capitalized) are
        # checked directly -- no "_r" OpenFace-style suffix guessing needed here
        # since these are confirmed exact Detectorv2 column names.
        all_target_cols = TARGET_AUS + EXTRA_AFFECT_COLS
        resolved_cols = {}
        for col in all_target_cols:
            if col in openface_df.columns:
                resolved_cols[col] = col
        missing_cols = set(all_target_cols) - set(resolved_cols.keys())
        if missing_cols:
            print(f"  [!] WARNING: could not find column(s) {missing_cols} in {expected_openface_csv}. "
                  f"Actual columns: {list(openface_df.columns)}")

        trial_ppi = trial_ppi.sort_values("elapsed_trial_sec")

        # Correct the facial timestamp using the measured camera-start offset (see
        # camera_timing above) instead of assuming frame 0 == phase start.
        camera_offset_sec = 0.0
        if robot_name in camera_timing and not trial_ppi.empty:
            phase_start_wallclock = (trial_ppi["absolute_time"] - trial_ppi["elapsed_trial_sec"]).iloc[0]
            camera_offset_sec = camera_timing[robot_name] - phase_start_wallclock
            print(f"  [i] Measured camera-start offset for {robot_name}: {camera_offset_sec:.3f}s")

        if "frame" in openface_df.columns:
            openface_df["timestamp"] = camera_offset_sec + openface_df["frame"] / ACTUAL_FPS
        elif "timestamp" in openface_df.columns:
            openface_df["timestamp"] = camera_offset_sec + openface_df["timestamp"]
        else:
            print(f"  [!] WARNING: no 'frame' or 'timestamp' column in {expected_openface_csv} -- "
                  f"cannot time-align facial data. Actual columns: {list(openface_df.columns)}")
            trial_integrated_dfs.append(trial_ppi)
            continue

        openface_df = openface_df.sort_values("timestamp")

        # Keep only what we actually need from the facial side before merging
        facial_cols_to_keep = ["timestamp"] + list(resolved_cols.values())
        if conf_col:
            facial_cols_to_keep.append(conf_col)
        openface_slim = openface_df[facial_cols_to_keep]

        merged_trial = pd.merge_asof(
            trial_ppi,
            openface_slim,
            left_on="elapsed_trial_sec",
            right_on="timestamp",
            direction="nearest",
            tolerance=MERGE_TOLERANCE_SEC,
        )

        trial_integrated_dfs.append(merged_trial)

    if not trial_integrated_dfs:
        print(f"[!] Error: No trial data could be processed for participant {PARTICIPANT_ID}.")
        continue

    all_trials_timeseries = pd.concat(trial_integrated_dfs, ignore_index=True)

    # ==========================================
    # STEP B: PROCESS & MERGE GODSPEED SCORES
    # ==========================================
    print("\nProcessing and merging Godspeed questionnaire ratings...")

    clean_scores_df = process_godspeed_scores(scores_df, PARTICIPANT_ID, godspeed_file)

    if not clean_scores_df.empty:
        subscale_output_filename = f"participant_{PARTICIPANT_ID}_godspeed_subscales.csv"
        clean_scores_df.to_csv(subscale_output_filename, index=False)
        print(f"  [+] Calculated subscales saved to: {subscale_output_filename}")
    else:
        print(f"  [!] WARNING: no Godspeed subscales were computed -- {godspeed_file} may be "
              f"empty or missing a 'robot_name' column.")

    if "trial_name" in all_trials_timeseries.columns and "robot_name" in all_trials_timeseries.columns:
        all_trials_timeseries = all_trials_timeseries.drop(columns=["trial_name"])

    if not clean_scores_df.empty and "participant_id" in all_trials_timeseries.columns and "participant_id" in clean_scores_df.columns:
        individual_master_df = pd.merge(
            all_trials_timeseries,
            clean_scores_df,
            on=["participant_id", "robot_name"],
            how="left"
        )
    else:
        individual_master_df = all_trials_timeseries

    # ==========================================
    # STEP C: SAVE INDIVIDUAL MASTER FILE
    # ==========================================
    output_filename = f"participant_{PARTICIPANT_ID}_master_integrated.csv"
    individual_master_df.to_csv(output_filename, index=False)
    print(f"Success! Individual master file saved as: {output_filename}\n")

    # ==========================================
    # STEP D: TRIAL-LEVEL SUMMARY (RMSSD + AU/affect averages + Godspeed)
    # ==========================================
    # Everything above produces raw, per-timestep data. The actual comparisons this
    # thesis is built around (RMSSD and AU/affect measures across the five
    # conditions, correlated with Godspeed) need one row per trial, not one row
    # per timestep -- this step builds that.
    print("Building trial-level summary (RMSSD + AU/affect averages + Godspeed)...")

    rmssd_df = pd.DataFrame(trial_rmssd_rows)

    summary_cols_present = [c for c in (TARGET_AUS + EXTRA_AFFECT_COLS) if c in all_trials_timeseries.columns]
    if summary_cols_present:
        affect_summary_df = all_trials_timeseries.groupby(["participant_id", "robot_name"])[summary_cols_present].mean().reset_index()
        affect_summary_df = affect_summary_df.rename(columns={c: f"{c}_mean" for c in summary_cols_present})
    else:
        print("  [!] WARNING: no AU/affect columns found in the merged data -- these will be missing "
              f"from the trial summary. Expected some of {TARGET_AUS + EXTRA_AFFECT_COLS}.")
        affect_summary_df = pd.DataFrame(columns=["participant_id", "robot_name"])

    trial_summary_df = rmssd_df.merge(affect_summary_df, on=["participant_id", "robot_name"], how="left")
    if not clean_scores_df.empty:
        trial_summary_df = trial_summary_df.merge(clean_scores_df, on=["participant_id", "robot_name"], how="left")

    summary_output_filename = f"participant_{PARTICIPANT_ID}_trial_summary.csv"
    trial_summary_df.to_csv(summary_output_filename, index=False)
    print(f"Success! Trial-level summary saved as: {summary_output_filename}")
    print(trial_summary_df.to_string(index=False))

print(f"\n{'='*70}")
print(f"ALL {len(PARTICIPANT_IDS)} PARTICIPANTS PROCESSED.")
print(f"{'='*70}")

In [ ]:
import pandas as pd

df_master = pd.read_csv("participant_5_master_integrated.csv")
df_summary = pd.read_csv("participant_5_trial_summary.csv")

print("=== AU11 check ===")
print(df_master["AU11"].describe())
print("Unique AU11 values:", sorted(df_master["AU11"].unique())[:10])
print()

print("=== Any NaN in key columns? ===")
key_cols = ["AU04","AU05","AU07","AU09","arousal","valence","FaceScore","ppi_ms"]
print(df_master[key_cols].isna().sum())
print()

print("=== trial_summary.csv, full table ===")
print(df_summary[["robot_name","baseline_rmssd","stimulus_rmssd","rmssd_delta",
                   "AU04_mean","AU05_mean","AU07_mean","AU09_mean","AU11_mean",
                   "arousal_mean","valence_mean"]].to_string(index=False))

In [ ]:
df = pd.read_csv("participant_4_master_integrated.csv")
for robot, g in df.groupby("robot_name"):
    n_total = len(g)
    n_unique_facial = g["timestamp"].nunique()
    print(f"{robot}: {n_total} PPI rows -> {n_unique_facial} unique facial timestamps matched")

In [ ]:
import subprocess

for robot in ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]:
    video = f"participant_4_{robot}_cam.avi"
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=r_frame_rate,nb_frames,duration",
         "-of", "default=noprint_wrappers=1", video],
        capture_output=True, text=True
    )
    print(f"--- {robot} ---")
    print(result.stdout)

In [ ]:
import pandas as pd

# Check how many frames actually got dropped in participant 4
df_raw = pd.read_csv("participant_4_Robot1_openface.csv")
below_threshold = (df_raw["FaceScore"] < 0.75).sum()
print(f"Robot1: {len(df_raw)} total frames, {below_threshold} below 0.75 threshold ({below_threshold/len(df_raw)*100:.1f}%)")

In [ ]:
import pandas as pd

df = pd.read_csv("participant_4_continuous_ppi.csv")
video_ppi = df[df["phase"] == "video_playback"].copy()

for robot, g in video_ppi.groupby("trial_name" if "trial_name" in video_ppi.columns else "robot_name"):
    g = g.sort_values("elapsed_trial_sec")
    gaps = g["elapsed_trial_sec"].diff().dropna()
    print(f"--- {robot} ---")
    print(f"  median gap: {gaps.median():.3f}s, min: {gaps.min():.3f}s, max: {gaps.max():.3f}s")
    print(f"  gaps < 0.05s (near-simultaneous): {(gaps < 0.05).sum()} / {len(gaps)}")

In [ ]:
import pandas as pd

df = pd.read_csv("participant_4_master_integrated.csv")

au_cols = [c for c in df.columns if c.startswith("AU")] + ["arousal", "valence",
           "Neutral","Happy","Sad","Surprise","Fear","Disgust","Anger"]

# Old way (PPI-row-weighted, biased by burst clustering)
old_means = df.groupby("robot_name")[au_cols].mean()

# Corrected way: one row per unique facial timestamp before averaging
df_unique_facial = df.drop_duplicates(subset=["robot_name", "timestamp"])
corrected_means = df_unique_facial.groupby("robot_name")[au_cols].mean()

print("=== OLD (PPI-weighted) vs CORRECTED (facial-frame-weighted) — AU04 ===")
comparison = pd.DataFrame({"old_AU04_mean": old_means["AU04"], "corrected_AU04_mean": corrected_means["AU04"]})
print(comparison)

In [ ]:
import inspect
import feat
print(feat.__version__)
from feat import Detector
print(inspect.signature(Detector.detect))

In [ ]:
import inspect
import feat
print([n for n in dir(feat) if "etector" in n])
from feat import Detectorv2
print(inspect.signature(Detectorv2.detect))